# 5 -- Construcción y definición del grafo convolucional  T-GCN — Dataset FDIC RIS

En los notebooks anteriores hemos construido dos estructuras fundamentales:

- Los **embeddings TabPFN** $e_i^t \in \mathbb{R}^{192}$, que condensan la historia tabular CAMELS de cada banco $i$ en el trimestre $t$ en un vector latente extraído por un transformer in-context.
- Los **snapshots del grafo dinámico** $\{G^1, \dots, G^{23}\}$, donde cada $G^t = (V^t, E^t, X^t, A^t)$ representa la red de conglomerados financieros bancarios en el trimestre $t$, con atributos nodales $x_i^t = [e_i^t \Vert s_i^t] \in \mathbb{R}^{204}$.

Este notebook entrena el **T-GCN** (Zhao et al., 2019) sobre el bloque de desarrollo (2016Q2 → 2021Q4). El modelo aprende a predecir qué bancos quiebran en el siguiente trimestre combinando dos fuentes de información que ningún modelo tabular puede capturar simultáneamente:

1. **Dependencias espaciales**: la posición de un banco en la red de holdings modula su riesgo. Un banco sano dentro de un conglomerado en dificultades tiene un perfil de riesgo distinto a un banco sano aislado, aunque sus ratios CAMELS sean idénticos.
2. **Dependencias temporales**: la evolución de esa posición a lo largo de los trimestres anteriores, capturada por el estado oculto GRU.

La hipótesis central es que el riesgo bancario tiene una **componente sistémica no capturada por features individuales**, observable en la estructura de relaciones entre entidades.

## Estructura del notebook

1. Configuración y paths
2. Carga de snapshots
3. Split temporal train/val
4. Análisis del desbalanceo de clases
5. Arquitectura T-GCN: revisión y justificación matemática
6. Función de pérdida: BCE ponderada
7. Bucle de entrenamiento
8. Métricas de evaluación
9. Curvas de aprendizaje
10. Ajuste de hiperparámetros
11. Selección y guardado del mejor modelo

Resumen de lo que reutilizamos y lo que descartamos:
De la implementación de referencia reutilizamos únicamente calculate_laplacian_with_self_loop de graph_conv.py y la lógica de TGCNCell (las dos TGCNGraphConvolution para puertas rr
r y uu
u). Todo lo demás — pytorch-lightning, tasks, callbacks, losses, métricas — se descarta porque es infraestructura para forecasting de tráfico por regresión.
Las tres diferencias de adaptación críticas:

Input multidimensional. El repo original asume xit∈Rx_i^t \in \mathbb{R}
xit​∈R (velocidad de tráfico escalar). Nosotros tenemos xit∈R204x_i^t \in \mathbb{R}^{204}
xit​∈R204. Esto cambia el shape de la concatenación [x,h][x, h]
[x,h] en TGCNGraphConvolution: en vez de (num_nodes, hidden_dim + 1) tenemos (num_nodes, hidden_dim + input_dim), y la matriz de pesos WW
W pasa de R(dh+1)×dh\mathbb{R}^{(d_{h}+1) \times d_{h}}
R(dh​+1)×dh​ a R(dh+dx)×dh\mathbb{R}^{(d_{h}+d_{x}) \times d_{h}}
R(dh​+dx​)×dh​.
Laplaciano dinámico. El repo lo registra fijo en __init__ porque el grafo de tráfico no cambia. Nosotros calculamos L^t\hat{L}^t
L^t en cada paso temporal del forward porque AtA^t
At varía por trimestre.
Clasificación binaria nodo a nodo. La salida del repo es (batch_size, num_nodes, hidden_dim) seguida de un regresor lineal. Nosotros añadimos una capa Linear(hidden_dim, 1) seguida de sigmoid para obtener y^it∈[0,1]\hat{y}_i^t \in [0,1]
y^​it​∈[0,1] por nodo, con BCE ponderada como loss inicial.


Antes de escribir el código, una decisión que necesito que confirmes:
¿Cómo tratamos los nodos aislados (bancos sin holding, \sim$22.7% del grafo)? Con $\hat{L}^t = \tilde{D}^{-1/2}\tilde{A}\tilde{D}^{-1/2}
 y A~=A+I\tilde{A} = A + I
A~=A+I, un nodo aislado tiene únicamente la auto-conexión, así que su agregación GCN es simplemente su propio estado proyectado. No hay problema matemático — el modelo los trata como nodos con vecindario N(i)={i}\mathcal{N}(i) = \{i\}
N(i)={i}.

Un nodo aislado con A~ii=1\tilde{A}_{ii}=1
A~ii​=1 y grado d~i=1\tilde{d}_i=1
d~i​=1 produce L^ii=1\hat{L}_{ii}=1
L^ii​=1, por lo que la agregación GCN se reduce a:
Hi(1)=σ(xitW)H_i^{(1)} = \sigma(x_i^t W)Hi(1)​=σ(xit​W)
Es decir, una transformación lineal del propio estado sin propagación de vecinos. Esto es correcto y tiene interpretación directa: un banco independiente (sin holding) no recibe señal de contagio estructural, solo transforma sus propias features. Es exactamente el comportamiento que queremos modelar — la ausencia de aristas no es un artefacto, es información semántica real.
En la memoria se justifica así: los nodos aislados en GtG^t
Gt corresponden a bancos sin RSSDHCR, es decir, entidades independientes no pertenecientes a ningún conglomerado financiero regulatorio. Para estos nodos, la convolución GCN degenera en una proyección lineal L^ii=1\hat{L}_{ii}=1
L^ii​=1, lo que equivale a afirmar que su riesgo relacional es función exclusiva de sus atributos individuales, sin componente de contagio sistémico. Esto es coherente con la definición de EtE^t
Et.

Puntos clave de la implementación:
calculate_laplacian — se llama una vez por snapshot en el forward, no en __init__. Coste O(N2)O(N^2)
O(N2) con N≈5.000N \approx 5.000
N≈5.000, perfectamente viable en GPU.
TGCNGraphConvolution — la concatenación es [h,x][h, x]
[h,x] de dimensión (dh+dx)(d_h + d_x)
(dh​+dx​) en vez de (dh+1)(d_h + 1)
(dh​+1) del repo original. W∈R(dh+dx)×output_dimW \in \mathbb{R}^{(d_h + d_x) \times \text{output\_dim}}
W∈R(dh​+dx​)×output_dim es la única diferencia estructural respecto a la referencia.
TGCNCell — graph_conv1 computa [z,r][z, r]
[z,r] conjuntamente con output_dim = 2 d_h y se separa con torch.chunk, igual que en la GRU estándar. bias=1.0 en las puertas inicializa el modelo en modo "conservar estado anterior", que es un punto de partida estable.
TGCN.forward — recibe una lista de snapshots ordenados temporalmente. Cuando Nt≠Nt−1N_t \neq N_{t-1}
Nt​=Nt−1​ el estado oculto se reinicializa. La alineación de nodos persistentes entre trimestres la gestiona el notebook 06, no el modelo. Los logits salen sin sigmoid para usar BCEWithLogitsLoss, que es numéricamente más estable que BCELoss(sigmoid(logits)).

## 1. Configuración y paths

In [ ]:
# ============================================================
# CELDA 1 — Configuración global
# ============================================================
import sys
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from torch_geometric.utils import to_dense_adj
from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    f1_score, precision_score, recall_score
)

# Paths — ajustar si la estructura de directorios difiere
ROOT   = Path('D:/financial_risk_data')
GRAPHS = ROOT / 'graphs'
MODELS = ROOT / 'models_checkpoints'
MODELS.mkdir(exist_ok=True)

# Añadir models/ al path para importar tgcn.py
sys.path.insert(0, str(ROOT / 'src'))
from models.tgcn import TGCN
from data.graph_builder import GraphBuilder

# Reproducibilidad
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

# Device: GPU si está disponible (Colab), CPU en local
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

## 2. Carga de snapshots

Los snapshots fueron construidos en el notebook 04 con `GraphBuilder.save_snapshots()`,
que serializa cada $G^t$ como un archivo `.pt` usando `torch.save`. Cada objeto
`torch_geometric.data.Data` contiene:

- `data.x` $\in \mathbb{R}^{|V^t| \times 204}$: atributos de nodo.
- `data.edge_index` $\in \mathbb{Z}^{2 \times |E^t|}$: aristas en formato COO.
- `data.y` $\in \{0,1\}^{|V^t|}$: etiquetas de quiebra.
- `data.cert`: lista de CERTs en el mismo orden que las filas de `data.x`.
- `data.period`: trimestre en formato `'YYYYQq'`.

Nótese que los snapshots **no almacenan la matriz de adyacencia densa** $A^t$
sino `edge_index` en formato COO para ahorrar memoria. La conversión a densa
se hace en el forward del modelo con `to_dense_adj`, justificada porque
$N \approx 5.000$ hace que $A^t \in \mathbb{R}^{5000 \times 5000}$ sea manejable
en GPU ($\sim$100 MB en float32).

In [ ]:
# ============================================================
# CELDA 2 — Carga de snapshots desarrollo
# ============================================================
snapshots_dev = GraphBuilder.load_snapshots(GRAPHS / 'snapshots_desarrollo')

print(f'\nResumen del bloque de desarrollo:')
print(f'  Snapshots cargados : {len(snapshots_dev)}')
print(f'  Rango temporal     : {snapshots_dev[0].period} → {snapshots_dev[-1].period}')
print(f'  dim_x              : {snapshots_dev[0].x.shape[1]}')
print()

# Verificación de integridad
for snap in snapshots_dev:
    assert snap.x.shape[1] == 204, f'dim_x inesperada en {snap.period}'
    assert snap.y.shape[0] == snap.num_nodes, f'Desajuste y/num_nodes en {snap.period}'

total_positivos = sum(int(s.y.sum()) for s in snapshots_dev)
total_nodos     = sum(s.num_nodes for s in snapshots_dev)
print(f'  Total nodos        : {total_nodos:,}')
print(f'  Total positivos    : {total_positivos}')
print(f'  Prevalencia global : {total_positivos/total_nodos*100:.4f}%')

## 3. Split temporal train / validación

La separación temporal es un requisito de validez del sistema de early warning:
el modelo debe entrenarse sobre historia pasada y evaluarse sobre periodos futuros
no vistos, replicando las condiciones reales de despliegue.

Un split aleatorio introduciría **leakage temporal**: el modelo vería el estado
del grafo en $t+k$ durante el entrenamiento, lo que inflaría artificialmente
las métricas y produciría un sistema inútil en producción.

Elegimos la frontera en **2020Q3** por dos razones:

1. **Proporcional**: 17 snapshots de train (74%) y 6 de validación (26%).
2. **Económica**: separa el periodo pre-COVID del post-COVID. El shock sistémico
   de 2020 es el evento de estrés más relevante del bloque; el modelo debe
   aprenderlo desde el contexto pre-shock y generalizarlo en validación.

Con 63 positivos en 23 trimestres, **walk-forward validation** (entrenar sobre
$t=1..k$, validar sobre $t=k+1$, avanzar) produciría folds con 0-3 positivos
cada uno, haciendo las métricas por fold inestables. El hold-out temporal
concentra ~13 positivos en validación, suficiente para estimar AUROC.

In [ ]:
# ============================================================
# CELDA 3 — Split temporal
# ============================================================
FRONTERA_VAL = '2020Q3'  # primer trimestre de validación

snapshots_train = [s for s in snapshots_dev if s.period < FRONTERA_VAL]
snapshots_val   = [s for s in snapshots_dev if s.period >= FRONTERA_VAL]

pos_train = sum(int(s.y.sum()) for s in snapshots_train)
pos_val   = sum(int(s.y.sum()) for s in snapshots_val)
n_train   = sum(s.num_nodes for s in snapshots_train)
n_val     = sum(s.num_nodes for s in snapshots_val)

print(f'Split temporal:')
print(f'  Train : {snapshots_train[0].period} → {snapshots_train[-1].period}')
print(f'          {len(snapshots_train)} snapshots | {n_train:,} nodos | {pos_train} positivos')
print(f'  Val   : {snapshots_val[0].period} → {snapshots_val[-1].period}')
print(f'          {len(snapshots_val)} snapshots | {n_val:,} nodos | {pos_val} positivos')

## 4. Desbalanceo de clases y función de pérdida

El desbalanceo es extremo: $\sim$0.044% de positivos globalmente. La función
de pérdida estándar BCE sin ponderar converge trivialmente prediciendo siempre 0,
obteniendo pérdida mínima sin aprender nada.

**BCE ponderada** (`BCEWithLogitsLoss` con `pos_weight`):

$$
\mathcal{L} = -\frac{1}{N} \sum_{i=1}^{N}
\left[ w^+ \cdot y_i \log \hat{p}_i + (1 - y_i) \log(1 - \hat{p}_i) \right]
$$

donde $w^+ = n_{\text{neg}} / n_{\text{pos}}$ amplifica el gradiente de los
positivos en proporción a su escasez. Con $w^+ \approx 2.270$, cada quiebra
contribuye al gradiente como si hubiera 2.270 ejemplos en vez de uno.

`BCEWithLogitsLoss` combina sigmoid y BCE en una sola operación numéricamente
estable usando el truco $\log(\sigma(x)) = -\log(1 + e^{-x})$, evitando
underflow/overflow para logits extremos. Por eso el modelo devuelve logits
crudos (sin sigmoid) y la pérdida aplica sigmoid internamente.

**Focal Loss** (Lin et al., 2017) queda como experimento posterior:
$$
\text{FL}(p_t) = -\alpha_t (1 - p_t)^\gamma \log(p_t)
$$
El factor $(1-p_t)^\gamma$ reduce el peso de negativos fáciles (bien clasificados)
y concentra el aprendizaje en ejemplos difíciles. Se comparará con BCE ponderada
en la sección de ajuste de hiperparámetros.

In [ ]:
# ============================================================
# CELDA 4 — Cálculo de pos_weight sobre el bloque de train
# ============================================================
# pos_weight se calcula SOLO sobre train para no contaminar con información
# de validación. Es un estadístico del conjunto de entrenamiento.

n_neg_train = n_train - pos_train
pos_weight  = n_neg_train / pos_train

print(f'Bloque train:')
print(f'  Negativos : {n_neg_train:,}')
print(f'  Positivos : {pos_train}')
print(f'  pos_weight: {pos_weight:.2f}')
print()
print(f'Interpretación: cada quiebra en train contribuye al gradiente')
print(f'como si hubiera {pos_weight:.0f} ejemplos en vez de uno.')

## 5. Arquitectura T-GCN — revisión y justificación

La arquitectura completa se define en `models/tgcn.py`. Aquí revisamos
las decisiones de diseño y sus fundamentos matemáticos.

### 5.1 Convolución GCN dentro de la celda GRU

La normalización simétrica del Laplaciano (Kipf & Welling, 2017):

$$
\hat{L}^t = \tilde{D}^{-1/2} \tilde{A}^t \tilde{D}^{-1/2},
\quad \tilde{A}^t = A^t + I
$$

garantiza dos propiedades:
- **Auto-conexión** ($+I$): el nodo incluye su propio estado en la agregación.
  Sin ella, la GCN ignora la información propia del nodo y solo propaga la de vecinos.
- **Invariancia al grado**: nodos con muchos vecinos (holdings grandes) no acumulan
  señal arbitrariamente grande. $\tilde{D}^{-1/2}$ normaliza simétricamente.

Para nodos aislados (bancos sin holding), $\tilde{A}^t_{ii} = 1$ y
$\tilde{D}^t_{ii} = 1$, por lo que $\hat{L}^t_{ii} = 1$. La convolución
degenera en la identidad: $h_i^{(1)} = \sigma(x_i^t W)$. Esto es correcto:
un banco independiente transforma solo sus propias features sin recibir
señal de contagio estructural.

### 5.2 Celda T-GCN

La GRU estándar sustituye las multiplicaciones matriciales lineales por
convoluciones GCN sobre el grafo en $t$:

$$
z_t = \sigma\bigl(\hat{L}^t [h_{t-1}, x_t] W_z + b_z\bigr)
\quad \text{(puerta de actualización)}
$$
$$
r_t = \sigma\bigl(\hat{L}^t [h_{t-1}, x_t] W_r + b_r\bigr)
\quad \text{(puerta de reset)}
$$
$$
\tilde{h}_t = \tanh\bigl(\hat{L}^t [r_t \odot h_{t-1}, x_t] W_h + b_h\bigr)
\quad \text{(candidato)}
$$
$$
h_t = (1 - z_t) \odot h_{t-1} + z_t \odot \tilde{h}_t
$$

La clave es que $[h_{t-1}, x_t]$ se concatena **antes** de la multiplicación
por $\hat{L}^t$, de modo que la propagación sobre el grafo actúa sobre
la representación conjunta de estado oculto y features de entrada.

### 5.3 Laplaciano dinámico

$\hat{L}^t$ se recalcula en cada paso temporal del forward porque $A^t$ varía
por trimestre (quiebras y fusiones modifican $V^t$ y $E^t$). Esto tiene coste
$O(N^2)$ por snapshot, con $N \approx 5.000$, perfectamente viable en GPU.

### 5.4 Cabeza clasificadora

Tras procesar la secuencia de snapshots, el estado oculto final
$h_i^T \in \mathbb{R}^{d_h}$ de cada nodo contiene su embedding dinámico:
incorpora la estructura relacional del grafo en $T$ (vía GCN) y la evolución
temporal de esa estructura (vía GRU). Una capa lineal proyecta a logit escalar:

$$
\ell_i = h_i^T W_c + b_c \in \mathbb{R}
$$
$$
\hat{y}_i = \sigma(\ell_i) = P(\text{quiebra}_i \mid G^1, \dots, G^T)
$$

In [ ]:
# ============================================================
# CELDA 5 — Instanciación del modelo y función de pérdida
# ============================================================
INPUT_DIM  = 204   # d_x = 192 (TabPFN) + 12 (estructurales)
HIDDEN_DIM = 64    # d_h — punto de partida conservador dado el nº de positivos

model = TGCN(input_dim=INPUT_DIM, hidden_dim=HIDDEN_DIM).to(DEVICE)

# Parámetros totales
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Modelo T-GCN instanciado:')
print(f'  input_dim  : {INPUT_DIM}')
print(f'  hidden_dim : {HIDDEN_DIM}')
print(f'  Parámetros : {n_params:,}')
print()

# BCE ponderada — pos_weight en el mismo device que el modelo
pos_weight_tensor = torch.tensor([pos_weight], dtype=torch.float32).to(DEVICE)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight_tensor)

print(f'Función de pérdida: BCEWithLogitsLoss(pos_weight={pos_weight:.2f})')

## 6. Estrategia de entrenamiento

### 6.1 Forward pass sobre la secuencia completa de train

En cada epoch, el modelo procesa la secuencia ordenada
$\{G^1_{\text{train}}, \dots, G^{17}_{\text{train}}\}$ snapshot a snapshot.
El estado oculto $h^t$ se propaga entre snapshots: el estado de $t-1$ inicializa
la celda GRU en $t$. La pérdida se acumula sobre todos los snapshots de train.

Formalmente, en cada epoch:
$$
\mathcal{L}_{\text{epoch}} = \frac{1}{T_{\text{train}}} \sum_{t=1}^{T_{\text{train}}}
\mathcal{L}_{\text{BCE}}(\hat{y}^t, y^t)
$$

donde $\hat{y}^t$ son los logits del snapshot $t$ y $y^t$ sus etiquetas.

### 6.2 Gestión del estado oculto entre snapshots

Cuando $|V^t| \neq |V^{t-1}|$ (un banco quiebra o aparece uno nuevo),
el estado oculto cambia de dimensión. La estrategia es:
**reinicializar $h$ a cero al inicio de cada epoch** y dejar que la GRU
lo construya progresivamente a lo largo de la secuencia temporal.

Esto es correcto porque el modelo aprende a iniciar el estado desde cero
en 2016Q2 y construirlo trimestre a trimestre, que es exactamente
lo que ocurrirá en el despliegue real.

### 6.3 Alineación de nodos entre snapshots

El estado oculto $h^t \in \mathbb{R}^{|V^t| \times d_h}$ tiene una fila
por nodo activo en $t$. Para pasarlo como inicialización en $t+1$, donde
$|V^{t+1}|$ puede diferir, el modelo reinicializa $h$ a cero cada vez que
$N_t$ cambia. Los nodos que permanecen entre trimestres "pierden" su estado
oculto en este diseño simple.

Una alternativa más sofisticada (alineación por CERT) se deja como mejora
futura: requeriría un índice de correspondencia entre filas de $G^t$ y $G^{t+1}$,
que es factible con `data.cert` pero añade complejidad de implementación.

### 6.4 Optimizador

Adam con lr=1e-3 y weight_decay=0 como punto de partida, igual que en
la implementación de referencia para T-GCN. Weight decay no nulo se
explorará en el ajuste de hiperparámetros.

In [ ]:
# ============================================================
# CELDA 6 — Optimizador y configuración de entrenamiento
# ============================================================
LR           = 1e-3
WEIGHT_DECAY = 0.0
MAX_EPOCHS   = 200
PATIENCE     = 20   # early stopping sobre val AUROC

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=LR,
    weight_decay=WEIGHT_DECAY
)

print(f'Optimizador: Adam(lr={LR}, weight_decay={WEIGHT_DECAY})')
print(f'Max epochs : {MAX_EPOCHS}')
print(f'Patience   : {PATIENCE} epochs sin mejora en val AUROC')

## 7. Funciones auxiliares

### 7.1 Conversión edge_index → adyacencia densa

Los snapshots almacenan aristas en formato COO (`edge_index`). El forward del
T-GCN necesita la matriz $A^t$ densa para calcular el Laplaciano. La función
`to_dense_adj` de PyTorch Geometric hace esta conversión en $O(N^2)$.

### 7.2 Métricas de clasificación con desbalanceo extremo

Con $\sim$0.044% de positivos, la accuracy es completamente inútil (predecir
siempre 0 da accuracy > 99.95%). Las métricas relevantes son:

- **AUROC**: área bajo la curva ROC. Mide la capacidad discriminativa global
  independientemente del umbral de decisión. Valor esperado aleatorio: 0.5.
- **Average Precision (AUPRC)**: área bajo la curva Precision-Recall.
  Más informativa que AUROC con desbalanceo extremo porque penaliza
  fuertemente los falsos positivos. Valor esperado aleatorio: prevalencia ≈ 0.00044.
- **F1** con umbral optimizado sobre validación: $F_1 = 2 \cdot \frac{P \cdot R}{P + R}$.
  El umbral se elige maximizando $F_1$ sobre el conjunto de validación,
  no se fija a 0.5 (que sería subóptimo con este desbalanceo).

In [ ]:
# ============================================================
# CELDA 7 — Funciones auxiliares
# ============================================================
def snapshot_to_device(snapshot, device):
    """Mueve los tensores de un snapshot al device indicado."""
    snapshot.x          = snapshot.x.to(device)
    snapshot.edge_index = snapshot.edge_index.to(device)
    snapshot.y          = snapshot.y.to(device)
    return snapshot


def compute_adj_dense(snapshot):
    """
    Convierte edge_index COO a matriz de adyacencia densa A^t ∈ R^{N×N}.
    to_dense_adj devuelve (1, N, N); hacemos squeeze para obtener (N, N).
    """
    adj = to_dense_adj(
        snapshot.edge_index,
        max_num_nodes=snapshot.num_nodes
    ).squeeze(0)  # (N, N)
    return adj


def evaluate(model, snapshots, device, threshold=0.5):
    """
    Evalúa el modelo sobre una secuencia de snapshots.

    Procesa la secuencia completa en modo inferencia y recoge
    probabilidades y etiquetas de TODOS los snapshots (no solo el último).
    Esto permite calcular métricas agregadas sobre todo el bloque.

    Returns dict con auroc, auprc, f1, precision, recall.
    """
    model.eval()
    all_probs  = []
    all_labels = []

    with torch.no_grad():
        # Procesar snapshot a snapshot para recoger predicciones de cada uno
        hidden_state = None
        for snap in snapshots:
            snap    = snapshot_to_device(snap, device)
            adj     = compute_adj_dense(snap)   # (N, N)
            x       = snap.x                    # (N, d_x)
            y       = snap.y                    # (N,)
            n       = snap.num_nodes

            if hidden_state is None or hidden_state.size(0) != n:
                hidden_state = torch.zeros(n, model.hidden_dim, device=device)

            from models.tgcn import calculate_laplacian
            laplacian    = calculate_laplacian(adj)
            hidden_state, _ = model.tgcn_cell(x, hidden_state, laplacian)
            logits       = model.classifier(hidden_state).squeeze(1)
            probs        = torch.sigmoid(logits)

            all_probs.append(probs.cpu().numpy())
            all_labels.append(y.cpu().numpy())

    all_probs  = np.concatenate(all_probs)
    all_labels = np.concatenate(all_labels)

    auroc  = roc_auc_score(all_labels, all_probs)
    auprc  = average_precision_score(all_labels, all_probs)
    preds  = (all_probs >= threshold).astype(int)
    f1     = f1_score(all_labels, preds, zero_division=0)
    prec   = precision_score(all_labels, preds, zero_division=0)
    rec    = recall_score(all_labels, preds, zero_division=0)

    return {'auroc': auroc, 'auprc': auprc, 'f1': f1,
            'precision': prec, 'recall': rec}


def find_best_threshold(model, snapshots, device, thresholds=None):
    """
    Encuentra el umbral de clasificación que maximiza F1 sobre
    el conjunto dado. Se usa SOLO sobre validación, nunca sobre test.
    """
    if thresholds is None:
        thresholds = np.linspace(0.01, 0.99, 99)

    model.eval()
    all_probs, all_labels = [], []

    with torch.no_grad():
        hidden_state = None
        for snap in snapshots:
            snap = snapshot_to_device(snap, device)
            adj  = compute_adj_dense(snap)
            n    = snap.num_nodes
            if hidden_state is None or hidden_state.size(0) != n:
                hidden_state = torch.zeros(n, model.hidden_dim, device=device)
            from models.tgcn import calculate_laplacian
            laplacian       = calculate_laplacian(adj)
            hidden_state, _ = model.tgcn_cell(snap.x, hidden_state, laplacian)
            logits          = model.classifier(hidden_state).squeeze(1)
            all_probs.append(torch.sigmoid(logits).cpu().numpy())
            all_labels.append(snap.y.cpu().numpy())

    all_probs  = np.concatenate(all_probs)
    all_labels = np.concatenate(all_labels)

    best_f1, best_thr = 0.0, 0.5
    for thr in thresholds:
        f1 = f1_score(all_labels, (all_probs >= thr).astype(int), zero_division=0)
        if f1 > best_f1:
            best_f1, best_thr = f1, thr

    return best_thr, best_f1


print('Funciones auxiliares definidas.')

## 8. Bucle de entrenamiento

El bucle procesa la secuencia de snapshots de train en orden temporal,
acumulando pérdida y propagando gradientes al final de cada epoch.
El estado oculto se propaga entre snapshots dentro de cada epoch pero
se reinicializa al inicio de cada epoch nueva (`.detach()` para cortar
el grafo computacional y no acumular gradientes entre epochs).

**Early stopping**: se monitoriza el AUROC sobre validación. Si no mejora
durante `PATIENCE` epochs consecutivas, se detiene el entrenamiento y
se restauran los pesos del mejor epoch.

In [ ]:
# ============================================================
# CELDA 8 — Bucle de entrenamiento con early stopping
# ============================================================
import copy, time

history = {
    'train_loss': [], 'val_auroc': [], 'val_auprc': [], 'val_f1': []
}

best_val_auroc  = -1.0
best_epoch      = 0
best_model_wts  = copy.deepcopy(model.state_dict())
epochs_no_improv = 0

t0 = time.perf_counter()

for epoch in range(1, MAX_EPOCHS + 1):

    # ── TRAIN ─────────────────────────────────────────────────────────────
    model.train()
    optimizer.zero_grad()

    epoch_loss   = 0.0
    hidden_state = None

    for snap in snapshots_train:
        snap = snapshot_to_device(snap, DEVICE)
        adj  = compute_adj_dense(snap)   # (N, N)
        x    = snap.x                    # (N, d_x)
        y    = snap.y                    # (N,)
        n    = snap.num_nodes

        # Reinicializar estado si N cambia (quiebras/fusiones entre trimestres)
        # detach() corta el grafo computacional para no propagar gradientes
        # más allá del snapshot actual — equivalente a TBPTT con longitud 1.
        if hidden_state is None or hidden_state.size(0) != n:
            hidden_state = torch.zeros(n, model.hidden_dim, device=DEVICE)
        else:
            hidden_state = hidden_state.detach()

        from models.tgcn import calculate_laplacian
        laplacian       = calculate_laplacian(adj)
        hidden_state, _ = model.tgcn_cell(x, hidden_state, laplacian)
        logits          = model.classifier(hidden_state).squeeze(1)

        loss         = criterion(logits, y)
        epoch_loss  += loss.item()
        loss.backward()

    # Gradient clipping: evita explosión de gradientes en la GRU.
    # Norma máxima 1.0 es estándar para RNNs (Pascanu et al., 2013).
    nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
    optimizer.step()
    optimizer.zero_grad()

    avg_train_loss = epoch_loss / len(snapshots_train)

    # ── VALIDACIÓN ────────────────────────────────────────────────────────
    val_metrics = evaluate(model, snapshots_val, DEVICE)

    history['train_loss'].append(avg_train_loss)
    history['val_auroc'].append(val_metrics['auroc'])
    history['val_auprc'].append(val_metrics['auprc'])
    history['val_f1'].append(val_metrics['f1'])

    # ── EARLY STOPPING ────────────────────────────────────────────────────
    if val_metrics['auroc'] > best_val_auroc:
        best_val_auroc  = val_metrics['auroc']
        best_epoch      = epoch
        best_model_wts  = copy.deepcopy(model.state_dict())
        epochs_no_improv = 0
    else:
        epochs_no_improv += 1

    # ── LOG ───────────────────────────────────────────────────────────────
    if epoch % 10 == 0 or epoch == 1:
        elapsed = time.perf_counter() - t0
        print(
            f'Epoch {epoch:4d}/{MAX_EPOCHS} | '
            f'loss={avg_train_loss:.4f} | '
            f'val_auroc={val_metrics["auroc"]:.4f} | '
            f'val_auprc={val_metrics["auprc"]:.4f} | '
            f'val_f1={val_metrics["f1"]:.4f} | '
            f'elapsed={elapsed:.0f}s'
        )

    if epochs_no_improv >= PATIENCE:
        print(f'\nEarly stopping en epoch {epoch}. '
              f'Mejor epoch: {best_epoch} (val_auroc={best_val_auroc:.4f})')
        break

# Restaurar pesos del mejor epoch
model.load_state_dict(best_model_wts)
print(f'\nEntrenamiento completado. Mejor epoch: {best_epoch} | AUROC: {best_val_auroc:.4f}')

## 9. Curvas de aprendizaje

Las curvas de aprendizaje permiten diagnosticar el comportamiento del modelo:

- **Underfitting**: loss de train no decrece → modelo demasiado simple o lr muy bajo.
- **Overfitting**: train loss baja pero val AUROC no mejora o empeora → regularizar
  (weight decay, dropout) o reducir hidden_dim.
- **Comportamiento esperado**: train loss decrece monotónamente; val AUROC sube
  y se estabiliza. Con 63 positivos y grafo disperso, es posible que el AUROC
  sea modesto en esta configuración base.

In [ ]:
# ============================================================
# CELDA 9 — Curvas de aprendizaje
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

epochs_range = range(1, len(history['train_loss']) + 1)

axes[0].plot(epochs_range, history['train_loss'], label='Train loss', color='steelblue')
axes[0].axvline(best_epoch, color='red', linestyle='--', alpha=0.7, label=f'Best epoch ({best_epoch})')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('BCE Loss (ponderada)')
axes[0].set_title('Pérdida de entrenamiento')
axes[0].legend()

axes[1].plot(epochs_range, history['val_auroc'],  label='Val AUROC',  color='darkorange')
axes[1].plot(epochs_range, history['val_auprc'],  label='Val AUPRC',  color='green', linestyle='--')
axes[1].plot(epochs_range, history['val_f1'],     label='Val F1',     color='purple', linestyle=':')
axes[1].axvline(best_epoch, color='red', linestyle='--', alpha=0.7, label=f'Best epoch ({best_epoch})')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Métrica')
axes[1].set_title('Métricas de validación')
axes[1].legend()

plt.tight_layout()
plt.savefig(ROOT / 'graphs' / 'tgcn_learning_curves.png', dpi=150)
plt.show()

# Métricas finales del mejor epoch
best_metrics = evaluate(model, snapshots_val, DEVICE)
best_thr, best_f1 = find_best_threshold(model, snapshots_val, DEVICE)
print(f'\nMétricas del mejor modelo (epoch {best_epoch}):')
print(f'  AUROC     : {best_metrics["auroc"]:.4f}')
print(f'  AUPRC     : {best_metrics["auprc"]:.4f}')
print(f'  F1 (thr=0.5) : {best_metrics["f1"]:.4f}')
print(f'  Umbral óptimo: {best_thr:.3f} → F1={best_f1:.4f}')

## 10. Ajuste de hiperparámetros

Realizamos una búsqueda en grid sobre los hiperparámetros más influyentes.
El criterio de selección es **val AUROC** del mejor epoch (con early stopping).

**Espacio de búsqueda:**

| Hiperparámetro | Valores | Justificación |
|---|---|---|
| `hidden_dim` | 32, 64, 128 | Capacidad del modelo; 64 es punto de partida |
| `lr` | 1e-3, 5e-4 | Rango estándar para Adam en GNNs |
| `weight_decay` | 0.0, 1e-4 | Regularización L2; 0 es referencia |
| `pos_weight` | calculado, ×0.5, ×2 | Sensibilidad al desbalanceo |

Con 3×2×2×3 = 36 combinaciones y `MAX_EPOCHS=200`, la búsqueda completa
es costosa. En Colab con GPU A100 (~2 min/run) son ~72 min. Se puede
reducir a `MAX_EPOCHS=100` durante la búsqueda y refinar el mejor candidato
después con `MAX_EPOCHS=300`.

In [ ]:
# ============================================================
# CELDA 10 — Grid search de hiperparámetros
# ============================================================
import itertools

# Grid de búsqueda
param_grid = {
    'hidden_dim'   : [32, 64, 128],
    'lr'           : [1e-3, 5e-4],
    'weight_decay' : [0.0, 1e-4],
    'pw_factor'    : [0.5, 1.0, 2.0],  # multiplicador sobre pos_weight base
}

MAX_EPOCHS_SEARCH = 100   # reducido para la búsqueda
PATIENCE_SEARCH   = 15

keys   = list(param_grid.keys())
combos = list(itertools.product(*param_grid.values()))
print(f'Combinaciones a explorar: {len(combos)}')

results = []

for i, combo in enumerate(combos):
    params = dict(zip(keys, combo))
    pw     = pos_weight * params['pw_factor']

    # Instanciar modelo y criterio frescos
    m = TGCN(input_dim=INPUT_DIM, hidden_dim=params['hidden_dim']).to(DEVICE)
    c = nn.BCEWithLogitsLoss(
        pos_weight=torch.tensor([pw], dtype=torch.float32).to(DEVICE)
    )
    opt = torch.optim.Adam(
        m.parameters(), lr=params['lr'], weight_decay=params['weight_decay']
    )

    best_auroc_run = -1.0
    best_wts_run   = copy.deepcopy(m.state_dict())
    no_improv      = 0

    for epoch in range(1, MAX_EPOCHS_SEARCH + 1):
        m.train()
        opt.zero_grad()
        hidden_state = None

        for snap in snapshots_train:
            snap = snapshot_to_device(snap, DEVICE)
            adj  = compute_adj_dense(snap)
            n    = snap.num_nodes
            if hidden_state is None or hidden_state.size(0) != n:
                hidden_state = torch.zeros(n, m.hidden_dim, device=DEVICE)
            else:
                hidden_state = hidden_state.detach()
            from models.tgcn import calculate_laplacian
            laplacian       = calculate_laplacian(adj)
            hidden_state, _ = m.tgcn_cell(snap.x, hidden_state, laplacian)
            logits          = m.classifier(hidden_state).squeeze(1)
            loss            = c(logits, snap.y)
            loss.backward()

        nn.utils.clip_grad_norm_(m.parameters(), max_norm=1.0)
        opt.step()

        val_m = evaluate(m, snapshots_val, DEVICE)
        if val_m['auroc'] > best_auroc_run:
            best_auroc_run = val_m['auroc']
            best_wts_run   = copy.deepcopy(m.state_dict())
            no_improv      = 0
        else:
            no_improv += 1
        if no_improv >= PATIENCE_SEARCH:
            break

    results.append({**params, 'pw': pw, 'best_auroc': best_auroc_run})
    print(f'[{i+1:2d}/{len(combos)}] {params} | pw={pw:.1f} | AUROC={best_auroc_run:.4f}')

# Tabla de resultados ordenada
import pandas as pd
df_results = pd.DataFrame(results).sort_values('best_auroc', ascending=False)
print('\nTop 5 configuraciones:')
print(df_results.head(5).to_string(index=False))

## 11. Entrenamiento final y guardado del mejor modelo

Con los hiperparámetros óptimos identificados en la búsqueda, entrenamos
el modelo final con `MAX_EPOCHS=300` y guardamos:

- Los pesos del mejor epoch (`tgcn_best.pt`).
- El umbral de clasificación óptimo (`threshold_opt`).
- Los hiperparámetros (`tgcn_config.json`).

Estos artefactos son los que se cargarán en el notebook 07 para la evaluación
final sobre el bloque de evaluación (2022Q1 → 2025Q4).

In [ ]:
# ============================================================
# CELDA 11 — Entrenamiento final con mejores hiperparámetros
# ============================================================
import json

best_config = df_results.iloc[0].to_dict()
print(f'Mejor configuración encontrada:')
for k, v in best_config.items():
    print(f'  {k}: {v}')

# Modelo final
model_final = TGCN(
    input_dim=INPUT_DIM,
    hidden_dim=int(best_config['hidden_dim'])
).to(DEVICE)

criterion_final = nn.BCEWithLogitsLoss(
    pos_weight=torch.tensor([best_config['pw']], dtype=torch.float32).to(DEVICE)
)
optimizer_final = torch.optim.Adam(
    model_final.parameters(),
    lr=best_config['lr'],
    weight_decay=best_config['weight_decay']
)

MAX_EPOCHS_FINAL = 300
PATIENCE_FINAL   = 25

best_auroc_final = -1.0
best_epoch_final = 0
best_wts_final   = copy.deepcopy(model_final.state_dict())
no_improv_final  = 0
history_final    = {'train_loss': [], 'val_auroc': [], 'val_auprc': []}

for epoch in range(1, MAX_EPOCHS_FINAL + 1):
    model_final.train()
    optimizer_final.zero_grad()
    epoch_loss   = 0.0
    hidden_state = None

    for snap in snapshots_train:
        snap = snapshot_to_device(snap, DEVICE)
        adj  = compute_adj_dense(snap)
        n    = snap.num_nodes
        if hidden_state is None or hidden_state.size(0) != n:
            hidden_state = torch.zeros(n, model_final.hidden_dim, device=DEVICE)
        else:
            hidden_state = hidden_state.detach()
        from models.tgcn import calculate_laplacian
        laplacian       = calculate_laplacian(adj)
        hidden_state, _ = model_final.tgcn_cell(snap.x, hidden_state, laplacian)
        logits          = model_final.classifier(hidden_state).squeeze(1)
        loss            = criterion_final(logits, snap.y)
        epoch_loss     += loss.item()
        loss.backward()

    nn.utils.clip_grad_norm_(model_final.parameters(), max_norm=1.0)
    optimizer_final.step()
    optimizer_final.zero_grad()

    val_m = evaluate(model_final, snapshots_val, DEVICE)
    history_final['train_loss'].append(epoch_loss / len(snapshots_train))
    history_final['val_auroc'].append(val_m['auroc'])
    history_final['val_auprc'].append(val_m['auprc'])

    if val_m['auroc'] > best_auroc_final:
        best_auroc_final = val_m['auroc']
        best_epoch_final = epoch
        best_wts_final   = copy.deepcopy(model_final.state_dict())
        no_improv_final  = 0
    else:
        no_improv_final += 1

    if epoch % 25 == 0 or epoch == 1:
        print(f'Epoch {epoch:4d} | loss={history_final["train_loss"][-1]:.4f} | '
              f'val_auroc={val_m["auroc"]:.4f} | val_auprc={val_m["auprc"]:.4f}')

    if no_improv_final >= PATIENCE_FINAL:
        print(f'Early stopping en epoch {epoch}.')
        break

model_final.load_state_dict(best_wts_final)
print(f'\nMejor epoch final: {best_epoch_final} | AUROC: {best_auroc_final:.4f}')

In [ ]:
# ============================================================
# CELDA 12 — Guardado del modelo y artefactos
# ============================================================

# Umbral óptimo sobre validación
opt_threshold, opt_f1 = find_best_threshold(model_final, snapshots_val, DEVICE)

# Métricas finales con umbral óptimo
final_metrics = evaluate(model_final, snapshots_val, DEVICE, threshold=opt_threshold)

print(f'Métricas finales sobre validación (umbral={opt_threshold:.3f}):')
for k, v in final_metrics.items():
    print(f'  {k}: {v:.4f}')

# Guardar pesos
torch.save(model_final.state_dict(), MODELS / 'tgcn_best.pt')
print(f'\nPesos guardados en {MODELS / "tgcn_best.pt"}')

# Guardar configuración completa
config = {
    'input_dim'     : INPUT_DIM,
    'hidden_dim'    : int(best_config['hidden_dim']),
    'lr'            : best_config['lr'],
    'weight_decay'  : best_config['weight_decay'],
    'pos_weight'    : best_config['pw'],
    'best_epoch'    : best_epoch_final,
    'val_auroc'     : best_auroc_final,
    'val_auprc'     : final_metrics['auprc'],
    'val_f1'        : final_metrics['f1'],
    'opt_threshold' : opt_threshold,
    'frontera_val'  : FRONTERA_VAL,
    'seed'          : SEED,
}

with open(MODELS / 'tgcn_config.json', 'w') as f:
    json.dump(config, f, indent=2)

print(f'Configuración guardada en {MODELS / "tgcn_config.json"}')
print('\nArtefactos listos para notebook 07 (evaluación final).')